# Course 1 · Week 2 — Hands-on: Multivariate Regression & Feature Scaling

In Week 1 you fit a line to one feature. Now you'll fit a hyperplane to **three features at once** — and discover why feature scaling matters so much it's almost cheating not to use it.

By the end you'll have:

1. Vectorized prediction, cost, and gradient functions (work for any number of features)
2. Run gradient descent on raw, unscaled data and watched it crawl or diverge
3. Implemented z-score normalization
4. Re-run gradient descent on scaled features and watched it converge instantly
5. (Stretch) Built a cubic polynomial fit by adding `x²` and `x³` as new features

Estimated time: **45–60 minutes.**


## Setup — three features, wildly different scales

We're predicting house prices from three features:

- **Size** in square feet, range 800–2200
- **Bedrooms**, range 1–5
- **Age** in years, range 0–50

The size dimension is hundreds of times bigger than the bedrooms dimension. That mismatch is what makes feature scaling necessary — you'll see why concretely.

We know the true relationship (we made it up): `price = 0.15·size + 25·bedrooms − 1.5·age + 50` with some noise. Job: recover those coefficients from data alone.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)
m = 30  # number of houses

# Three features with deliberately different scales — this is the whole point of this notebook.
size_sqft = np.random.uniform(800, 2200, m)            # 800–2200 sq ft
bedrooms  = np.random.randint(1, 6, m).astype(float)    # 1–5 bedrooms
age_yrs   = np.random.uniform(0, 50, m)                  # 0–50 years old

# True price = 0.15*size + 25*bedrooms - 1.5*age + 50  (in $1000s) + noise
y = 0.15 * size_sqft + 25 * bedrooms - 1.5 * age_yrs + 50 + np.random.normal(0, 15, m)

X = np.column_stack([size_sqft, bedrooms, age_yrs])  # shape (30, 3)
print(f"X.shape = {X.shape}")
print(f"feature ranges:  size {size_sqft.min():.0f}–{size_sqft.max():.0f},  "
      f"bedrooms {bedrooms.min():.0f}–{bedrooms.max():.0f},  age {age_yrs.min():.0f}–{age_yrs.max():.0f}")
print(f"price range:    ${y.min():.0f}k – ${y.max():.0f}k")


Three quick scatter plots — one per feature against price. You can see size and bedrooms are positively correlated with price; age is weakly negative. None of these by itself fully explains the price; we need all three together.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
labels = ["size (sq ft)", "bedrooms", "age (years)"]
for i, ax in enumerate(axes):
    ax.scatter(X[:, i], y, color="#4338ca", s=60, alpha=0.7)
    ax.set_xlabel(labels[i])
    ax.set_ylabel("price ($1000s)")
    ax.grid(alpha=0.3)
plt.suptitle("Price vs each feature", y=1.02)
plt.tight_layout()
plt.show()


## Quick recap

The model now takes a vector of features instead of one number:

$$f_{w,b}(\vec x) = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b = \vec w \cdot \vec x + b$$

Cost is exactly the same form as Week 1, just with `f` taking a vector. Gradients are too, but `dJ/dw` is now a *vector* (one entry per feature) instead of a scalar.

The big new idea this week: **vectorization**. Instead of looping over examples, we use numpy's matrix operations to compute everything at once. Faster code AND fewer bugs.


## Exercise 1 — vectorized predict

Implement `predict(X, w, b)` so it works for any number of examples and any number of features.

Math: `X @ w + b`. Numpy's `@` is matrix multiplication. The `+ b` broadcasts.


In [ ]:
def predict(X, w, b):
    """Vectorized linear model.

    Args:
        X: numpy array, shape (m, n) — m examples, n features.
        w: numpy array, shape (n,) — one weight per feature.
        b: scalar.

    Returns:
        Array of predictions, shape (m,).
    """
    # TODO: write the one-line vectorized form. Hint: numpy's @ operator does matrix-vector multiply.
    return np.zeros(len(X))


# Sanity check (3 examples, 3 features, simple weights)
X_test = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=float)
w_test = np.array([1.0, 0.5, 0.0])
b_test = 10.0
out = predict(X_test, w_test, b_test)
expected = np.array([12.0, 16.5, 21.0])  # 1+1+0+10, 4+2.5+0+10, 7+4+0+10
print(f"predict output = {out}")
print(f"expected       = {expected}")
assert np.allclose(out, expected), "predict() should compute X @ w + b"
print("✓ predict() works")


## Exercise 2 — cost (same as Week 1, vectorized)

Same formula: `J(w, b) = (1/2m) · sum((yhat - y)²)`. With numpy you can compute the whole thing in one or two lines.


In [ ]:
def cost(X, y, w, b):
    """Mean squared error (with 1/2 factor)."""
    # TODO: implement using your predict() function.
    return 0.0


J0 = cost(X, y, np.zeros(3), 0.0)
print(f"cost at zeros = {J0:.2f}   (expected ≈ 52,376.30)")
assert abs(J0 - 52376.3039) < 1.0
print("✓ cost() works")


## Exercise 3 — vectorized gradients

For multi-feature regression, `dJ/dw` is now a *vector* with one entry per feature.

The formula in summation notation:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (f(\vec x^{(i)}) - y^{(i)}) \, x^{(i)}_j$$

In numpy, the whole gradient vector is:

```python
dw = (X.T @ err) / m
```

Why `X.T`? Because for each `j`, you want to multiply `err` by the `j`-th column of `X` and sum. That's exactly what `X.T @ err` does.


In [ ]:
def gradients(X, y, w, b):
    """Compute the partial derivatives.

    dJ/dw is a vector of length n: dJ/dw_j = (1/m) * sum( (yhat - y) * X[:, j] )
    dJ/db is a scalar:              dJ/db   = (1/m) * sum(  yhat - y         )

    Hint: with numpy, you can compute dw in one line as X.T @ err / m.
    """
    # TODO: implement.
    dw = np.zeros(X.shape[1])
    db = 0.0
    return dw, db


dw0, db0 = gradients(X, y, np.zeros(3), 0.0)
print(f"dw at zeros = {np.round(dw0, 2)}   (expected ≈ [-501738, -1107, -8649])")
print(f"db at zeros = {db0:.2f}        (expected ≈ -317.91)")
assert abs(dw0[0] - (-501738.47)) < 1.0
assert abs(db0 - (-317.91)) < 0.1
print("✓ gradients() works")


## Exercise 4 — gradient descent (the same skeleton as Week 1)

Implement `gradient_descent` with the **simultaneous update** discipline. You already wrote this once; the only difference is `w` is now a vector.

Then run it on the **raw, unscaled** features. Try several learning rates and watch what happens. Keep the rates very small — the gradients are huge.


In [ ]:
def gradient_descent(X, y, w0, b0, alpha, n_iters):
    """Run gradient descent for n_iters iterations.

    Returns (w, b, history) where history is the cost at each iteration.
    """
    w = w0.copy()
    b = b0
    history = []
    for _ in range(n_iters):
        # TODO: compute gradients, then simultaneously update w and b, then record cost
        pass
    return w, b, history


# Try gradient descent on the RAW (unscaled) features. Try several learning rates.
print("Without feature scaling:")
for alpha in [1e-7, 1e-6, 1e-5, 1e-4]:
    w, b, hist = gradient_descent(X, y, np.zeros(3), 0.0, alpha, 1000)
    if hist and np.isfinite(hist[-1]) and hist[-1] < 1e8:
        print(f"  alpha={alpha:>7}: final cost {hist[-1]:.2f}")
    else:
        print(f"  alpha={alpha:>7}: DIVERGED ❌")


## Exercise 5 — feature scaling (z-score normalization)

For each column `j`, compute the mean `μ_j` and standard deviation `σ_j`, then transform:

$$X'_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}$$

After this, every feature has mean 0 and std 1. Now they're all on the same scale, and a single learning rate can serve every feature.


In [ ]:
def zscore(X):
    """Z-score normalize each column independently.

    Returns:
        X_scaled: same shape as X, but each column has mean 0 and std 1.
        mu:       mean of each column (length n).
        sigma:    std of each column  (length n).
    """
    # TODO: implement using X.mean and X.std along axis=0
    return X, np.zeros(X.shape[1]), np.ones(X.shape[1])


X_scaled, mu, sigma = zscore(X)
print(f"After scaling — should be near 0 and 1:")
print(f"  mean of each column: {np.round(X_scaled.mean(axis=0), 3)}")
print(f"  std  of each column: {np.round(X_scaled.std(axis=0), 3)}")
assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-6)
assert np.allclose(X_scaled.std(axis=0), 1, atol=1e-6)
print("✓ zscore() works")


## Exercise 6 — gradient descent with scaled features

Now run gradient descent on `X_scaled`. Try alphas `0.001, 0.01, 0.1` — values that *all* worked great even on Week 1's tiny single-feature problem. After scaling, they work here too.


In [ ]:
print("With feature scaling — same alphas range now works much better:")
for alpha in [0.001, 0.01, 0.1]:
    w, b, hist = gradient_descent(X_scaled, y, np.zeros(3), 0.0, alpha, 1000)
    print(f"  alpha={alpha:>5}: final cost {hist[-1]:.2f}")

# Pick the best one, plot the cost curve and report final parameters
w_final, b_final, hist = gradient_descent(X_scaled, y, np.zeros(3), 0.0, alpha=0.1, n_iters=1000)
print(f"\nFinal (in SCALED space):")
print(f"  w = {np.round(w_final, 3)}")
print(f"  b = {b_final:.3f}")
print(f"  final cost = {hist[-1]:.2f}")

plt.figure(figsize=(7, 4))
plt.plot(hist, color="#4338ca")
plt.xlabel("iteration")
plt.ylabel("cost J")
plt.yscale("log")
plt.title("Cost curve with feature scaling, alpha=0.1")
plt.grid(alpha=0.3)
plt.show()


## Exercise 7 — predict a new house

Pretend a new house comes in: 1500 sq ft, 3 bedrooms, 10 years old. What price?

You **must** scale this new house's features using the same `mu` and `sigma` you computed from training data. If you don't, the model — which only ever saw scaled inputs during training — will give nonsense.


In [ ]:
# Predict on a new house. CRITICAL: scale the new house's features using the SAME mu and sigma you computed from the training data.
new_house = np.array([1500, 3, 10])  # 1500 sq ft, 3 bedrooms, 10 years old
# TODO: scale new_house using mu and sigma, then predict
new_scaled = new_house  # ← replace this with the scaled version
prediction = 0.0        # ← replace with predict(new_scaled, w_final, b_final)

print(f"new house features (scaled): {np.round(new_scaled, 3)}")
print(f"predicted price: ${prediction:.1f}k")


## ⭐ Stretch — polynomial regression

We've been fitting *linear* models — straight lines or hyperplanes. But real data is often curved. What if you wanted to fit `y = 0.5x³ − 2x + 1` (a cubic)?

The trick: **engineer new features**. Define a new dataset whose three columns are `x, x², x³`, and fit a *linear* model to those columns. The same gradient descent code works — it can't tell that you've smuggled in a cubic. Suddenly your linear model can fit a curve.

Try fitting:
- (a) just `x` — should fit poorly (a straight line through cubic data)
- (b) `x, x², x³` — should fit much better

Compare the final costs.


In [ ]:
# STRETCH: Polynomial regression
# Fit y = 0.5 x^3 - 2x + 1 (cubic) with two models:
#   (a) just x as a feature  — should fit poorly
#   (b) x, x^2, x^3 as features — should fit much better

np.random.seed(1)
m_p = 30
x_p = np.linspace(-3, 3, m_p)
y_p = 0.5 * x_p**3 - 2 * x_p + 1 + np.random.normal(0, 1.5, m_p)

# TODO (a): build X_lin with just x as a single feature. Scale it. Fit. Compute final cost.
# TODO (b): build X_poly with three features [x, x**2, x**3]. Scale. Fit. Compute final cost.
# Print the two costs to compare.


## Wrap-up

Two big ideas this week:

1. **Vectorization.** The same algorithm scales from 1 to 1,000,000 features without any loop changes — numpy and matrix math handle it. Every modern ML library is built on this.
2. **Feature scaling.** When features have wildly different scales, gradient descent fights itself. Scale them and the same algorithm just works. In the real world, *almost always* scale your features unless you have a very specific reason not to.

[Next: Week 3 — Classification](../exercises/course1-week3-classification.ipynb).
